# PR NO. 5 (LP IV) - Continuous Bag of Words (CBOW) Model
Cleaned-up code with an explanation for every block

**AIM:** Build a CBOW model that learns word embeddings by predicting a *centre* word from the words around it (its context).

**OVERALL FLOW:** Prepare text -> Tokenize -> Generate (context, target) pairs -> Build model -> Compile -> Train -> Predict -> Inspect embeddings

**KEY IDEA:** Skip-gram predicts the surrounding words from the centre word. CBOW does the opposite: the embeddings of the context words are **averaged** into one vector, which is used to predict the centre word. Word order inside the window is ignored, hence "Bag of Words".

**STAGES:** a. Data preparation | b. Generate training data | c. Train model | d. Output


# STAGE A - Data preparation
Import libraries, define the corpus, and convert words to integer IDs (Blocks 1-3).

## BLOCK 1 - Import libraries
**WHAT:** Imports everything the experiment needs.

**WHY:**
- `numpy` : array maths; `random` : fixing the random seed
- `tensorflow / keras` : building and training the model
- `Tokenizer` : converts words to integer IDs
- `Sequential` : layers stacked one after another; `Input` : declares the input shape
- `Embedding` : lookup table turning a word ID into a dense vector (this is what CBOW learns)
- `GlobalAveragePooling1D` : averages the context word vectors
- `Dense` : fully connected output layer

**NOTE:** The original used a `Lambda` layer for averaging. `GlobalAveragePooling1D` does the same job (mean over context words) and is safer to save/load.

In [ ]:
import numpy as np
import random
import tensorflow as tf

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer

# Fix random seeds so results are (mostly) reproducible
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

## BLOCK 2 - Prepare the text (corpus)
**WHAT:** Defines the training text and splits it into sentences.

**WHY:** CBOW needs text to learn from. We keep each sentence separate so a context window never mixes words from two different sentences.

**NOTE:** The original joined everything into one long sequence, so windows such as (`learning`, `machine`) crossed a sentence boundary. Those pairs are meaningless, so we avoid them.

In [ ]:
text = """I love machine learning
I love deep learning
machine learning is interesting
deep learning is powerful"""

sentences = text.split("\n")
print("Sentences:", sentences)

## BLOCK 3 - Tokenize (word -> integer ID)
**WHAT:** Builds the vocabulary and converts each sentence to a list of IDs.

**WHY:**
- `fit_on_texts` learns the vocabulary (lower-cases words, so "I" and "i" are the same).
- `texts_to_sequences` replaces every word by its integer ID.
- `word_index` is `{word: id}`. IDs start at 1; ID 0 is reserved for padding.
- `vocab_size = len(word_index) + 1` : the +1 accounts for the unused ID 0.
- `reverse_word` is `{id: word}`, used to turn a predicted ID back into a word.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
word_index = tokenizer.word_index
reverse_word = {v: k for k, v in word_index.items()}
vocab_size = len(word_index) + 1

print("Vocabulary:", word_index)
print("Vocab size (incl. padding id 0):", vocab_size)
print("Sequences :", sequences)

# STAGE B - Generate training data
Slide a window over each sentence to build (context words -> centre word) pairs (Block 4).

## BLOCK 4 - Generate training data (context -> centre word)
**WHAT:** Slides a window over each sentence and creates (context words, centre word) pairs.

**WHY:**
- `window = 1` means 1 word on the left and 1 on the right, so every input has `2 * window = 2` context words.
- Example: "I love machine learning" -> centre `love` has context `[i, machine]`; centre `machine` has context `[love, learning]`.
- `X` = context IDs, shape `(num_samples, 2 * window)`; `y` = centre word ID.

**NOTE:** The original one-hot encoded `y` with `to_categorical`. We keep integers and use `sparse_categorical_crossentropy` later: same result, less code.

In [ ]:
window = 1
X, y = [], []

for seq in sequences:
    for i in range(window, len(seq) - window):
        context = seq[i - window:i] + seq[i + 1:i + window + 1]
        X.append(context)
        y.append(seq[i])

X = np.array(X)
y = np.array(y)

print("X (context IDs):\n", X)
print("y (centre IDs) :", y)
print("Samples:", len(X))

# STAGE C - Train model
Build the CBOW network, compile it, and train it (Blocks 5-7).

## BLOCK 5 - Build the CBOW model
**WHAT:** Defines the network layer by layer.

**WHY:**
- `Input(2 * window)` : each sample is 2 context word IDs.
- `Embedding(vocab_size, 10)` : turns each ID into a 10-number vector, output shape `(batch, 2, 10)`. These are the **word embeddings** the model learns.
- `GlobalAveragePooling1D` : averages the 2 context vectors into one, shape `(batch, 10)`. This is the "bag of words" step.
- `Dense(vocab_size, softmax)` : one probability per vocabulary word; the highest is the predicted centre word.

In [ ]:
embedding_dim = 10

model = Sequential([
    Input(shape=(2 * window,)),
    Embedding(vocab_size, embedding_dim),
    GlobalAveragePooling1D(),
    Dense(vocab_size, activation="softmax"),
])

## BLOCK 6 - Compile the model and view summary
**WHAT:** Chooses the optimizer, loss and metric, then prints the architecture.

**WHY:**
- optimizer `adam` : adaptive optimizer, works well with little tuning.
- loss `sparse_categorical_crossentropy` : multi-class loss when labels are integers.
- metric `accuracy` : % of centre words predicted correctly.
- `model.summary()` : layer shapes and parameter counts (Embedding = `vocab_size x 10`).

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## BLOCK 7 - Train the model
**WHAT:** Trains for 200 epochs and prints the final loss and accuracy.

**WHY:**
- `epochs=200` : the dataset is tiny, so many passes are needed for the embeddings to settle.
- `verbose=0` : hides per-epoch output.
- `history` : stores per-epoch loss/accuracy.

**NOTE:** There is no validation split because the data is so small. High accuracy only shows the model memorised the corpus; it is a demonstration, not a benchmark.

In [ ]:
history = model.fit(X, y, epochs=200, verbose=0)

print(f"Final training loss    : {history.history['loss'][-1]:.4f}")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")

# STAGE D - Output
Predict centre words from context, inspect the learned embeddings, and print the result (Blocks 8-11).

## BLOCK 8 - Helper function to predict the centre word
**WHAT:** Defines `predict_word()`, which takes context words and returns the most likely centre words with probabilities.

**WHY:**
- Converts words to IDs, runs `model.predict`, converts the best IDs back to words.
- Returns the top-k answers because several answers can be valid (see Block 9).
- Checks that words are in the vocabulary and that the number of context words equals `2 * window`.
- ID 0 is skipped because it is padding, not a real word.

In [ ]:
def predict_word(context_words, top_k=3):
    if len(context_words) != 2 * window:
        raise ValueError(f"Give exactly {2 * window} context words.")
    for w in context_words:
        if w.lower() not in word_index:
            raise ValueError(f"'{w}' is not in the vocabulary.")

    ids = np.array([[word_index[w.lower()] for w in context_words]])
    probs = model.predict(ids, verbose=0)[0]
    best = np.argsort(probs)[::-1][:top_k]
    return [(reverse_word[i], float(probs[i])) for i in best if i != 0]

## BLOCK 9 - Predict a centre word
**WHAT:** Predicts the missing word between "love" and "learning", then between "deep" and "is".

**WHY:**
- `["love", "learning"]` fits **two** training sentences ("I love machine learning" and "I love deep learning"), so "machine" and "deep" are both correct. Expect both to get a high, similar probability; which comes first can change between runs.
- `["deep", "is"]` appears only in "deep learning is", so "learning" should get a very high probability.
- Showing the top-3 makes this ambiguity visible; the original printed only one word.

In [ ]:
context = ["love", "learning"]
print("Context Words:", context)
for word, p in predict_word(context):
    print(f"  {word:<12} probability = {p:.3f}")

print()
context = ["deep", "is"]
print("Context Words:", context)
for word, p in predict_word(context):
    print(f"  {word:<12} probability = {p:.3f}")

## BLOCK 10 - Inspect the learned word embeddings
**WHAT:** Extracts the embedding matrix and compares word vectors using cosine similarity.

**WHY:**
- The real product of CBOW is the Embedding layer's weights: one 10-number vector per word.
- Cosine similarity (closer to 1 = more similar) shows which words the model treats as similar. Words in similar contexts (e.g. `machine` and `deep`, both between `love` and `learning`) should get similar vectors.
- With a tiny corpus the numbers are only illustrative.

In [ ]:
embeddings = model.layers[0].get_weights()[0]   # shape (vocab_size, 10)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Embedding of 'learning':\n", embeddings[word_index["learning"]])

pairs = [("machine", "deep"), ("machine", "interesting"), ("i", "love")]
for w1, w2 in pairs:
    sim = cosine_similarity(embeddings[word_index[w1]],
                            embeddings[word_index[w2]])
    print(f"Similarity({w1}, {w2}) = {sim:.3f}")

## BLOCK 11 - Conclusion
**WHAT:** Prints a summary using the real computed values.

**WHY:** An f-string always shows current numbers instead of a hard-coded comment that goes stale after retraining.

In [ ]:
print(f"The CBOW model (window={window}, embedding size={embedding_dim}) "
      f"reached training accuracy = {history.history['accuracy'][-1]:.4f} "
      f"on a vocabulary of {vocab_size - 1} words.")

## SUMMARY OF CHANGES FROM THE ORIGINAL NOTEBOOK
1. Added random seeds for reproducibility.
2. Tokenized sentence by sentence, so windows no longer cross sentence boundaries.
3. Replaced `Lambda(reduce_mean)` with `GlobalAveragePooling1D`.
4. Used the `Input(...)` layer to declare the input shape.
5. Kept labels as integers with `sparse_categorical_crossentropy` instead of one-hot.
6. Added `predict_word()` with input checks and top-3 probabilities.
7. Added an embedding inspection block with cosine similarity.
8. Replaced hard-coded output with a printed result.

**NOTE:** The original predicted "deep" for `["love", "learning"]`. "machine" is equally valid, so either may come out on top. Results vary slightly from run to run.